# DPI Probe Gate: stronger attacks, part B

Budget-matched control (best-of-48, no feedback) and feedback search in any language (Qwen3-8B). Synthetic data, dry-run. See attack.py.

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/code.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
!mkdir -p /kaggle/working/logs && df -h / /kaggle/working | tail -2

In [ ]:
!python attack.py paraphrase --n 48 --tag bo48 2>&1 | tee /kaggle/working/logs/bo48_paraphrase.log

In [ ]:
!python attack.py run --model qwen3-8b --tag bo48 --batch-size 1 2>&1 | tee /kaggle/working/logs/bo48_qwen.log

In [ ]:
!python attack.py search --model qwen3-8b --arm multi 2>&1 | tee /kaggle/working/logs/search_multi.log

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B ~/.cache/huggingface/hub/models--dheeyantra--dhee-pravega; df -h / | tail -1

In [ ]:
!pip install -q bitsandbytes
!python attack.py judge --model qwen3-8b --tag bo48 --load-4bit 2>&1 | tee /kaggle/working/logs/judge_bo48.log
!python attack.py judge --model qwen3-8b --tag search_multi --load-4bit 2>&1 | tee /kaggle/working/logs/judge_search_multi.log

In [ ]:
!zip -q -r /kaggle/working/attack2b_outputs.zip data/attack && ls -la /kaggle/working/attack2b_outputs.zip